# 16 · Localization 与 Mapping：从漂移到可恢复定位

感知模型输出目标之后，系统仍需要知道 ego vehicle 在地图和路网中的位置。L4 研发岗位通常同时关注 state estimation、地图匹配、传感器退化和 relocalization。

本 notebook 使用二维道路中心线作为简化地图，模拟：

- wheel/IMU odometry 的累积漂移；
- GNSS 噪声、outlier 和 outage；
- 基于地图最近点的 map matching；
- odometry + GNSS + map prior 的轻量融合。

这不是完整的 factor graph 或 LiDAR SLAM，但接口和误差分析与真实系统一致：明确坐标系、观测时间和退化模式。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

def make_route(steps=260, dt=0.1):
    t = np.arange(steps) * dt
    x = 0.65 * t
    y = 3.5 * np.sin(x / 14.0) + 0.8 * np.sin(x / 5.0)
    return t, np.c_[x, y]


time_s, truth = make_route()
map_x = np.linspace(truth[:, 0].min() - 3, truth[:, 0].max() + 3, 700)
map_points = np.c_[map_x, 3.5 * np.sin(map_x / 14.0) + 0.8 * np.sin(map_x / 5.0)]


## Part A — 观测模型与漂移

设真值位置为 \(p_t\)。里程计提供增量

\[
\Delta \hat p_t = \Delta p_t + b\Delta t + \epsilon_t,
\]

因而积分后的误差会随时间增长。GNSS 是绝对观测，但会出现噪声和离群点；地图匹配则提供一个带有地图先验的几何约束。


In [ ]:
def simulate_observations(gnss_noise=1.4, outage_start=120, outage_end=175, seed=12):
    rng = np.random.default_rng(seed)
    delta = np.diff(truth, axis=0, prepend=truth[:1])
    odom_delta = delta + np.array([0.012, -0.008]) + rng.normal(0, 0.035, delta.shape)
    odom = truth[0] + np.cumsum(odom_delta, axis=0)
    gnss = truth + rng.normal(0, gnss_noise, truth.shape)
    valid = np.ones(len(truth), dtype=bool)
    valid[outage_start:outage_end] = False
    outlier_ids = np.arange(55, len(truth), 83)
    gnss[outlier_ids] += np.array([7.0, -5.0])
    valid[outlier_ids] = False
    return odom, gnss, valid


def nearest_map_point(position):
    distance = np.linalg.norm(map_points - position, axis=1)
    return map_points[np.argmin(distance)]


def fuse_localization(odom, gnss, gnss_valid, gnss_gain=0.18, map_gain=0.08):
    estimate = np.zeros_like(odom)
    estimate[0] = odom[0]
    for i in range(1, len(odom)):
        prediction = estimate[i - 1] + (odom[i] - odom[i - 1])
        if gnss_valid[i]:
            prediction = (1 - gnss_gain) * prediction + gnss_gain * gnss[i]
        matched = nearest_map_point(prediction)
        estimate[i] = (1 - map_gain) * prediction + map_gain * matched
    return estimate


odom, gnss, valid = simulate_observations()
fused = fuse_localization(odom, gnss, valid)

def report(name, estimate):
    error = np.linalg.norm(estimate - truth, axis=1)
    return {"system": name, "RMSE_m": np.sqrt(np.mean(error ** 2)), "p95_m": np.percentile(error, 95), "max_m": error.max()}


display(pd.DataFrame([report("odometry", odom), report("fused", fused)]))


In [ ]:
fig, ax = plt.subplots()
ax.plot(truth[:, 0], truth[:, 1], label="ground truth", linewidth=3)
ax.plot(map_points[:, 0], map_points[:, 1], "--", label="map centerline", alpha=0.7)
ax.plot(odom[:, 0], odom[:, 1], label="odometry drift")
ax.plot(gnss[valid, 0], gnss[valid, 1], ".", label="valid GNSS", alpha=0.35)
ax.plot(fused[:, 0], fused[:, 1], label="fused estimate")
ax.set_aspect("equal")
ax.set_xlabel("x / m")
ax.set_ylabel("y / m")
ax.set_title("Localization with odometry drift and map prior")
ax.legend()
plt.show()


In [ ]:
def inspect_localization(gnss_noise=1.4, map_gain=0.08, outage_length=55):
    odom, gnss, valid = simulate_observations(gnss_noise=gnss_noise, outage_end=120 + int(outage_length))
    estimate = fuse_localization(odom, gnss, valid, map_gain=map_gain)
    error = np.linalg.norm(estimate - truth, axis=1)
    print({k: round(v, 3) for k, v in report("fused", estimate).items() if k != "system"})
    plt.plot(time_s, error, label="position error")
    plt.axvspan(120 * 0.1, (120 + outage_length) * 0.1, color="red", alpha=0.12, label="GNSS outage")
    plt.axhline(0.8, color="black", linestyle="--", label="ODD sigma threshold")
    plt.xlabel("time / s")
    plt.ylabel("position error / m")
    plt.legend()
    plt.show()


interact(
    inspect_localization,
    gnss_noise=FloatSlider(value=1.4, min=0.1, max=4.0, step=0.1),
    map_gain=FloatSlider(value=0.08, min=0.0, max=0.5, step=0.02),
    outage_length=FloatSlider(value=55, min=0, max=120, step=5),
)


### 练习

1. 把 map matching 改成带 heading 的最近车道匹配，拒绝横向误差过大的匹配；
2. 加入 IMU yaw bias，并报告 heading RMSE；
3. 统计 GNSS outage 结束后恢复到 0.8 m 以内需要多少秒；
4. 构造一个“地图版本错误但 GNSS 正常”的 failure case；
5. 把融合输出写成 `localization_state = {pose, covariance, map_version, timestamp}`，供后续 planner 使用。


## 完成标准

除了画出轨迹，还要记录 outage、outlier、地图匹配错误对 RMSE/p95/max error 的影响，并说明哪些误差会直接改变 ODD 状态。
